In [3]:
from pathlib import Path
import pandas as pd
import numpy as np


IDX = ROOT / "indexes"

files = {
    "W=0.1": IDX / "solmaz_table_sift_W0p100.csv",
    "W=0.3": IDX / "solmaz_table_sift_W0p300.csv",
    "W=0.5": IDX / "solmaz_table_sift_W0p500.csv",
    "W=1.0": IDX / "solmaz_table_sift_W1p000.csv",
}

rows = []

for setting, path in files.items():
    df = pd.read_csv(path)

    
    if "MaxRecall" in df.columns:
        mask = df["MaxRecall"].isna() & df["SearchRt0p99_ms"].notna()
        df.loc[mask, "MaxRecall"] = df.loc[mask, "SearchRt0p99_ms"]
        df.loc[mask, "MaxRecallEf"] = df.loc[mask, "SearchRt0p995_ms"]
        df.loc[mask, "SearchRt0p99_ms"] = np.nan
        df.loc[mask, "SearchRt0p995_ms"] = np.nan

    for method in ["DAPG", "PSEUDO_DAPG", "AWARE_PSEUDO_DAPG", "LSHAPG_BASE"]:
        sub = df[df["SolmazMethod"] == method]
        if len(sub) == 0:
            continue

        r = sub.iloc[0]
        rows.append({
            "Setting": setting,
            "Method": method,
            "MaxRecall": r["MaxRecall"],
            "Ef(MaxRecall)": r["MaxRecallEf"],
            "SearchRt@0.95(ms)": r["SearchRt0p95_ms"],
            "SearchRt@0.97(ms)": r["SearchRt0p97_ms"],
            "IndexingTime(s)": r["IndexingTime_s"],
            "InsertAvg(ms)": r["InsertAvg_ms"],
            "DeleteAvg(ms)": r["DeleteAvg_ms"],
        })

table = pd.DataFrame(rows)
display(table)

,Setting,Method,MaxRecall,Ef(MaxRecall),SearchRt@0.95(ms),SearchRt@0.97(ms),IndexingTime(s),InsertAvg(ms),DeleteAvg(ms)
0,W=0.1,DAPG,0.9779,1960.0,0.84798,1.67025,73.2018,0.669015,0.162040
1,W=0.1,PSEUDO_DAPG,0.9784,2000.0,0.49665,0.93099,73.2018,0.583105,0.031200
2,W=0.1,AWARE_PSEUDO_DAPG,0.9784,2000.0,0.53415,0.91273,73.2018,0.582125,0.025650
3,W=0.1,LSHAPG_BASE,0.9776,2060.0,0.88402,1.49328,84.6713,NaN,NaN
4,W=0.3,DAPG,0.9772,1780.0,1.59172,2.99459,125.4890,1.146180,0.187010
5,W=0.3,PSEUDO_DAPG,0.9779,1800.0,1.03554,1.79924,125.4890,1.216980,0.081660
6,W=0.3,AWARE_PSEUDO_DAPG,0.9779,1780.0,0.84754,1.52698,125.4890,1.073980,0.053890
7,W=0.3,LSHAPG_BASE,0.9778,1820.0,1.72666,3.22728,135.2070,NaN,NaN
8,W=0.5,DAPG,0.9776,1980.0,1.85726,2.93963,133.0140,1.548480,0.201085
9,W=0.5,PSEUDO_DAPG,0.9779,1920.0,0.94875,1.98935,133.0140,1.136330,0.066845


In [20]:
variant_table = pd.DataFrame([
    {
        "Variant": "DAPG",
        "Category": "Base method",
        "What changes": "Distance-aware pruned graph search",
        "Shared DAPG graph?": "Yes",
    },
    {
        "Variant": "PSEUDO-DAPG",
        "Category": "Traversal / entry selection",
        "What changes": "Uses multiple pseudo entry points",
        "Shared DAPG graph?": "Yes",
    },
    {
        "Variant": "AWARE-PSEUDO-DAPG",
        "Category": "Traversal / candidate awareness",
        "What changes": "Reranks LSH candidates and selects stronger entries",
        "Shared DAPG graph?": "Yes",
    },
    {
        "Variant": "ANCHOR-PSEUDO-DAPG",
        "Category": "Traversal / anchor entry",
        "What changes": "Uses precomputed anchors for entry selection",
        "Shared DAPG graph?": "Yes",
    },
    {
        "Variant": "LEARNED-ANCHOR-PSEUDO-DAPG",
        "Category": "Traversal / learned anchor entry",
        "What changes": "Uses learned/softmax anchor selection",
        "Shared DAPG graph?": "Yes",
    },
    {
        "Variant": "DAPG-Lazy",
        "Category": "Update maintenance",
        "What changes": "Defers repair after deletion",
        "Shared DAPG graph?": "Yes",
    },
    {
        "Variant": "DAPG-Batch",
        "Category": "Update maintenance",
        "What changes": "Amortizes pruning/repair per update block",
        "Shared DAPG graph?": "Yes",
    },
    {
        "Variant": "DAPG+Hybrid",
        "Category": "Auxiliary candidate generation",
        "What changes": "Adds an HNSW candidate layer",
        "Shared DAPG graph?": "adds auxiliary HNSW",
    },
])

display(variant_table)

variant_table.to_csv(OUT / "dapg_variant_categories.csv", index=False)

with open(OUT / "dapg_variant_categories.md", "w", encoding="utf-8") as f:
    f.write(variant_table.to_markdown(index=False))



,Variant,Category,What changes,Shared DAPG graph?
0,DAPG,Base method,Distance-aware pruned graph search,Yes
1,PSEUDO-DAPG,Traversal / entry selection,Uses multiple pseudo entry points,Yes
2,AWARE-PSEUDO-DAPG,Traversal / candidate awareness,Reranks LSH candidates and selects stronger en...,Yes
3,ANCHOR-PSEUDO-DAPG,Traversal / anchor entry,Uses precomputed anchors for entry selection,Yes
4,LEARNED-ANCHOR-PSEUDO-DAPG,Traversal / learned anchor entry,Uses learned/softmax anchor selection,Yes
5,DAPG-Lazy,Update maintenance,Defers repair after deletion,Yes
6,DAPG-Batch,Update maintenance,Amortizes pruning/repair per update block,Yes
7,DAPG+Hybrid,Auxiliary candidate generation,Adds an HNSW candidate layer,adds auxiliary HNSW


In [22]:
files = {
    "W=0.1": IDX / "solmaz_table_sift_W0p100.csv",
    "W=0.3": IDX / "solmaz_table_sift_W0p300.csv",
    "W=0.5": IDX / "solmaz_table_sift_W0p500.csv",
    "W=1.0": IDX / "solmaz_table_sift_W1p000.csv",
}

methods = [
    "DAPG",
    "PSEUDO_DAPG",
    "AWARE_PSEUDO_DAPG",
    "ANCHOR_PSEUDO_DAPG",
    "LEARNED_ANCHOR_PSEUDO_DAPG",
    "LSHAPG_BASE",
]

rows = []

for setting, path in files.items():
    df = pd.read_csv(path)

    # Fix shifted MaxRecall fields in these CSVs.
    mask = df["MaxRecall"].isna() & df["SearchRt0p99_ms"].notna()
    df.loc[mask, "MaxRecall"] = df.loc[mask, "SearchRt0p99_ms"]
    df.loc[mask, "MaxRecallEf"] = df.loc[mask, "SearchRt0p995_ms"]
    df.loc[mask, "SearchRt0p99_ms"] = np.nan
    df.loc[mask, "SearchRt0p995_ms"] = np.nan

    for method in methods:
        sub = df[df["SolmazMethod"] == method]
        if len(sub) == 0:
            continue

        r = sub.iloc[0]
        rows.append({
            "W": setting,
            "Method": method.replace("_", "-"),
            "MaxRecall": r["MaxRecall"],
            "Ef(MaxRecall)": r["MaxRecallEf"],
            "SearchRt@0.95(ms)": r["SearchRt0p95_ms"],
            "SearchRt@0.97(ms)": r["SearchRt0p97_ms"],
            "IndexingTime(s)": r["IndexingTime_s"],
            "InsertAvg(ms)": r["InsertAvg_ms"],
            "DeleteAvg(ms)": r["DeleteAvg_ms"],
        })

sensitivity = pd.DataFrame(rows)

display(sensitivity)

sensitivity.to_csv(OUT / "sift1m_matched_traversal_sensitivity.csv", index=False)

with open(OUT / "sift1m_matched_traversal_sensitivity.md", "w", encoding="utf-8") as f:
    f.write(sensitivity.to_markdown(index=False))



,W,Method,MaxRecall,Ef(MaxRecall),SearchRt@0.95(ms),SearchRt@0.97(ms),IndexingTime(s),InsertAvg(ms),DeleteAvg(ms)
0,W=0.1,DAPG,0.9779,1960.0,0.84798,1.67025,73.2018,0.669015,0.162040
1,W=0.1,PSEUDO-DAPG,0.9784,2000.0,0.49665,0.93099,73.2018,0.583105,0.031200
2,W=0.1,AWARE-PSEUDO-DAPG,0.9784,2000.0,0.53415,0.91273,73.2018,0.582125,0.025650
3,W=0.1,ANCHOR-PSEUDO-DAPG,0.9784,2000.0,2.92117,3.21738,73.2018,0.628240,0.066855
4,W=0.1,LEARNED-ANCHOR-PSEUDO-DAPG,0.9784,2000.0,3.09658,3.41692,73.2018,0.604545,0.041605
5,W=0.1,LSHAPG-BASE,0.9776,2060.0,0.88402,1.49328,84.6713,NaN,NaN
6,W=0.3,DAPG,0.9772,1780.0,1.59172,2.99459,125.4890,1.146180,0.187010
7,W=0.3,PSEUDO-DAPG,0.9779,1800.0,1.03554,1.79924,125.4890,1.216980,0.081660
8,W=0.3,AWARE-PSEUDO-DAPG,0.9779,1780.0,0.84754,1.52698,125.4890,1.073980,0.053890
9,W=0.3,ANCHOR-PSEUDO-DAPG,0.9779,1820.0,5.75774,6.45180,125.4890,1.048490,0.045620
